In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2_contingency, kruskal

In [ ]:
FILE_PATH = Path(
    "../data/raw/張瓊之_給馬老師資料20250925_V4(遺失值不用登錄成-1).xlsx"
)

demo = pd.read_excel(FILE_PATH, sheet_name="demographic")
mmse = pd.read_excel(FILE_PATH, sheet_name="MMSE longitudinal")
casi = pd.read_excel(FILE_PATH, sheet_name="CASI longitudinal")

print("demographic shape:", demo.shape)
print("MMSE shape:", mmse.shape)
print("CASI shape:", casi.shape)

In [ ]:
mmse = mmse.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MMSE(numerical)": "mmse",
    "CDR (scale)": "cdr_global",
    "CDR_M (scale)": "cdr_memory",
    "CDR_O (scale)": "cdr_orientation",
    "CDR_J (scale)": "cdr_judgment",
    "CDR_C (scale)": "cdr_community",
    "CDR_H (scale)": "cdr_home_hobbies",
    "CDR_P (scale)": "cdr_personal_care",
    "CDR-SOB(numerical)": "cdr_sob"
})

mmse.head()

In [ ]:
mmse["patient_id"].unique()
set1 = set([i for i in range(1,1302)])
output = set1 - set(mmse["patient_id"])
print("mmse資料表缺失的病人id:", output)

In [ ]:
casi = casi.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MENMA10": "casi_mental_manipulation",
    "ATTEN8": "casi_attention",
    "ORIEN18": "casi_orientation",
    "LTM10": "casi_long_term_memory",
    "STM12": "casi_short_term_memory",
    "ABSTR12": "casi_abstraction",
    "DRAW10": "casi_drawing",
    "ANML10": "casi_verbal_fluency",
    "LANG10": "casi_language",
    "Total": "casi_total"
})

casi.head()

In [ ]:
mmse_casi = mmse.merge(
    casi,
    on=["patient_id", "visit_date"],
    how="left"
)

mmse_casi.head()

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

df = mmse_casi.copy()

df["visit_date"] = pd.to_datetime(df["visit_date"])
df["mmse"] = pd.to_numeric(df["mmse"], errors="coerce")
df["casi"] = pd.to_numeric(df["casi_total"], errors="coerce")

# 判定每位病人的 CASI 是否完全缺失
casi_status = (
    df.groupby("patient_id")["casi"]
      .apply(lambda x: x.notna().any())
      .rename("has_any_casi")
      .reset_index()
)

# 加回原始資料
df = df.merge(casi_status, on="patient_id", how="left")

df["casi_complete_missing"] = (~df["has_any_casi"]).astype(int)

# 每位病人的第一筆有效 MMSE
baseline_mmse = (
    df.dropna(subset=["mmse"])
      .sort_values(["patient_id", "visit_date"])
      .groupby("patient_id", as_index=False)
      .first()
)

baseline_mmse.groupby("casi_complete_missing")["mmse"].agg(
    ["count", "mean", "std", "median", "min", "max"]
)

In [ ]:
383/ 1297

In [ ]:
df_has_casi = df[df["casi_complete_missing"] == 1]
df_has_casi["patient_id"].unique()

In [ ]:
group_missing = baseline_mmse.loc[
    baseline_mmse["casi_complete_missing"] == 1, "mmse"
].dropna()

group_other = baseline_mmse.loc[
    baseline_mmse["casi_complete_missing"] == 0, "mmse"
].dropna()

t_stat, p_value = ttest_ind(
    group_missing,
    group_other,
    equal_var=False
)

print(f"CASI 完全缺失組 n = {len(group_missing)}")
print(f"其餘病人組 n = {len(group_other)}")
print(f"Welch's t = {t_stat:.4f}")
print(f"p-value = {p_value:.4g}")